In [1]:
# will take a url, scrape its job description and make a resume via LateX code

In [14]:
import os
import subprocess
import shutil

from pydantic import Field, BaseModel
from langchain_google_genai import ChatGoogleGenerativeAI
from utils import count_token

from prompt_utils_resume import jd_text, project_experience, job_url

In [3]:
class LatexResume(BaseModel):
    latex_code: str = Field(description="latex code")
    company_name: str = Field(description="name of company")
    role_title: str = Field(description="job role title")
    req_id: str  = Field(description="Job requisition ID")
    
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash").with_structured_output(LatexResume)

595

In [5]:
# read Latex code

base_resume_filepath = r"D:\Code\ak\career_resumes_repo\industry_2025\1-page resume\ak_resume_1pager.tex"
cls_filepath = r"D:\Code\ak\career_resumes_repo\industry_2025\1-page resume\res.cls"

with open(base_resume_filepath, "r") as f:
    base_latex = f.read()

with open(cls_filepath, "r") as f:
    cls_latex = f.read()

In [18]:
print(f"JD = {count_token(jd_text)} tokens")
print(f"cls_file = {count_token(cls_latex)} tokens")
print(f"resume_file = {count_token(base_latex)} tokens")

JD = 595 tokens
cls_file = 7429 tokens
resume_file = 2118 tokens


In [7]:
prompt = f"""
YOU ARE GIVEN THESE:
- I pasted the LATEX CODE of my resume below. 
- I even pasted the cls code of the style file that I want you to STRICTLY use.
- For additional context I even gave you my PROJECT EXPERIENCE in case you need it. 

YOUR TASK:
 - Modify if any changes are needed in my resume to reflect the JOB DESCRIPTION, and give me the modified Latex code. 
 - Also give the company name, role title and job ID 

INSTRUCTIONS:
1) Remember that you don’t need to run the code. At any cost, use the res.cls file even if you run into errors. 
2) No need to embolden any tech key words. Bolding the side-headings is fine just like I have it currently. 
3) When using ampersand, use the Latex (\&).
4) When adding new skills  tailored to the, try to avoid adding absurdly impractical skills. Just look at my natural skillset and deviate somewhat from it so that it aligns with the Job description.
5) In the executive summary, if you have  to mention my PhD, mention 'PhD Background'. That way it doesn't explicitly say that I don't have a PhD yet while still conveying my experience.

############################
JOB DESCRIPTION:
{jd_text}
############################
LATEX CODE of the resume:
{base_latex}
############################
LATEX CODE of the cls style file :
{cls_latex}
############################
PROJECT EXPERIENCE:
{project_experience}
############################
"""

In [8]:
# llm invoke
from langchain_core.messages import HumanMessage
human_msg = HumanMessage(content=prompt)
messages = [human_msg]
response = llm.invoke(messages)

In [9]:
company_name = response.company_name
role_title = response.role_title
req_id  = response.req_id
latex_code = response.latex_code

In [10]:
base_path = r"D:\Code\ak\career_resumes_repo\industry_2025"
folder_path = os.path.join(base_path, company_name, f"{role_title}-{req_id}")
os.makedirs(folder_path, exist_ok=True)

In [11]:
# save latex files to the created folder
tex_file = os.path.join(folder_path, "ak_resume.tex")
with open(tex_file, "w") as f:
    f.write(latex_code)

shutil.copy(
    cls_filepath,
    folder_path
)    

'D:\\Code\\ak\\career_resumes_repo\\industry_2025\\CVS Health\\Senior Data Scientist - Agentic AI & Decision Intelligence-R0889533\\res.cls'

In [12]:
# run_latex
try:
    subprocess.run(
        ["pdflatex", "-output-directory", folder_path, tex_file],
        check=True, capture_output=True
    )
    print("resume generated")
except subprocess.CalledProcessError as e:
    print(f"Latex error: {e.stderr.decode()}")


resume generated


### ---------------------------------------------------------

## extract JD from link

In [15]:
from langchain_community.document_loaders import WebBaseLoader

# Set up the loader with a User-Agent to prevent blocks
loader = WebBaseLoader(
    job_url,
    header_template={
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
)

docs = loader.load()

# 1. Try to get the main body text
content = docs[0].page_content.strip()

# 2. If the body is empty or too short (common on Workday/React sites),
# grab the metadata description instead.
if len(content) < 200: 
    job_description = docs[0].metadata.get('description', '')
else:
    job_description = content

print(f"✅ Final JD length: {len(job_description)} characters.")

✅ Final JD length: 5113 characters.


In [17]:
print(job_description)

We’re building a world of health around every individual — shaping a more connected, convenient and compassionate health experience. At CVS Health®, you’ll be surrounded by passionate colleagues who care deeply, innovate with purpose, hold ourselves accountable and prioritize safety and quality in everything we do. Join us and be part of something bigger – helping to simplify health care one person, one family and one community at a time. Requisition Job Description Position Summary We are seeking a Sr. Data Scientist – Agentic AI &amp; Decision Intelligence to pioneer the next generation of analytics and data driven decision-making. This role will focus on building and scaling agentic AI frameworks, conversational analytics, and decision intelligence systems that transform how our business interacts with data. You will design and execute a customer-centric analytics strategy powered by AI agents, LLMs, and generative AI, ensuring insights are not just delivered, but acted upon. This i